In [2]:
import os
import pandas as pd
import numpy as np

import mne
from mne.preprocessing.nirs import optical_density

# ========= USER PATHS =========
# Point these to your files


NIRS_PATH = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\sessions\Sessions 11-11-25\PID003\2025-11-11_003\2025-11-11_003.snirf"      # Or NIRx folder path
PSY_PATH  = r"C:\Users\thiago-ext\Documents\FNIRS\psychopy\sessions\Sessions 11-11-25\PID003\enem_blocks_PID003_20251111_140525.csv"
# ==============================

def load_nirs_events(nirs_path):
    """Read Aurora (.snirf) or NIRx folder and return a DataFrame of events with 'label','onset_s'."""
    if os.path.isdir(nirs_path):
        # NIRx folder (older format)
        raw = mne.io.read_raw_nirx(nirs_path, preload=False, verbose=False)
    else:
        # SNIRF file (Aurora)
        raw = mne.io.read_raw_snirf(nirs_path, preload=False, verbose=False)

    # Prefer Annotations (string labels and onsets in seconds from recording start)
    ann = raw.annotations
    if len(ann) > 0:
        df = pd.DataFrame({
            "label": ann.description,
            "onset_s": ann.onset  # seconds
        }).sort_values("onset_s").reset_index(drop=True)
        # Filter only our task markers (optional; keep all if you want)
        return df

    # Fallback: stim channels (rare with Aurora string labels, but just in case)
    stim_picks = mne.pick_types(raw.info, stim=True)
    if len(stim_picks) > 0:
        events = mne.find_events(raw, verbose=False)
        # Here we only have integer codes; you can map them if you had a codebook
        df = pd.DataFrame(events, columns=["sample","prev","code"])
        df["onset_s"] = df["sample"] / raw.info["sfreq"]
        df["label"] = df["code"].astype(str)
        return df[["label","onset_s"]].sort_values("onset_s").reset_index(drop=True)

    raise RuntimeError("No annotations or stim channels found in the NIRS file.")

def load_psychopy_events(csv_path):
    """Return PsychoPy events with 'label','t_abs' (s) filtered to your task markers."""
    df = pd.read_csv(csv_path, encoding="utf-8")
    # Keep rows that are true markers of interest
    keep = df["marker_name"].astype(str).isin([
        "EXP_START","PING","QUESTIONNAIRE_ON","QNR_ITEM","QUESTIONNAIRE_OFF",
        "BLK_ON","BLK_OFF","BLOCK_REST","ITI",
        "Q_TEXT_ON","BUTTON_CLICK","Q_STEM_ON","Q_OPTIONS_ON",
        "ANS_A","ANS_B","ANS_C","ANS_D","ANS_E"
    ])
    df = df[keep].copy()
    df = df.rename(columns={"marker_name":"label","t_abs":"t_abs"})
    # Ensure numeric seconds
    df["t_abs"] = pd.to_numeric(df["t_abs"], errors="coerce")
    df = df.dropna(subset=["t_abs"])
    return df[["label","t_abs","block","trial_idx_in_block"]].sort_values("t_abs").reset_index(drop=True)

def choose_anchor(nirs_labels, psy_labels):
    """Pick a good first common label to align timelines."""
    preferred = ["PING","QUESTIONNAIRE_ON","BLK_ON","Q_TEXT_ON"]
    commons = [lab for lab in preferred if (lab in nirs_labels and lab in psy_labels)]
    if commons:
        return commons[0]
    # fallback to any first common label
    commons = list(set(nirs_labels).intersection(set(psy_labels)))
    if not commons:
        raise RuntimeError("No common labels between SNIRF annotations and PsychoPy log.")
    return sorted(commons)[0]

def align_and_compare(nirs_df, psy_df):
    # Pick anchor and compute offset: onset_snirf - t_abs_psy of the first occurrence
    anchor = choose_anchor(nirs_df["label"].tolist(), psy_df["label"].tolist())
    n0 = nirs_df[nirs_df["label"] == anchor].iloc[0]["onset_s"]
    p0 = psy_df[psy_df["label"] == anchor].iloc[0]["t_abs"]
    offset = n0 - p0

    # Shift PsychoPy times into the NIRS timeline
    psy_df = psy_df.copy()
    psy_df["psy_time_in_nirs_s"] = psy_df["t_abs"] + offset

    # Count comparison
    count_cmp = (
        pd.DataFrame(nirs_df["label"].value_counts(), columns=["count_snirf"])
        .join(psy_df["label"].value_counts().to_frame("count_psy"), how="outer")
        .fillna(0).astype(int).sort_index()
    )

    # Pairwise timing deltas (per label in order of appearance)
    # We’ll do a left-join per label, matching by order within each label.
    nirs_df["idx_in_label"] = nirs_df.groupby("label").cumcount()
    psy_df["idx_in_label"]  = psy_df.groupby("label").cumcount()

    merged = pd.merge(
        nirs_df, 
        psy_df[["label","psy_time_in_nirs_s","t_abs","block","trial_idx_in_block","idx_in_label"]],
        on=["label","idx_in_label"],
        how="outer",
        suffixes=("_snirf","_psy")
    ).sort_values(["onset_s","label","idx_in_label"])

    merged["time_delta_s"] = merged["onset_s"] - merged["psy_time_in_nirs_s"]

    return offset, anchor, count_cmp, merged

def quick_stats(merged):
    ok = merged.dropna(subset=["time_delta_s"])
    if len(ok) == 0:
        return "No aligned rows to compute timing deltas."
    abs_med = np.median(np.abs(ok["time_delta_s"]))
    abs_p95 = np.percentile(np.abs(ok["time_delta_s"]), 95)
    return f"|Δ| median = {abs_med:.003f}s, 95th = {abs_p95:.003f}s over {len(ok)} matched markers."

# ---- run
nirs_events = load_nirs_events(NIRS_PATH)
psy_events  = load_psychopy_events(PSY_PATH)

offset, anchor, counts, merged = align_and_compare(nirs_events, psy_events)

print("\n=== Alignment ===")
print(f"Anchor label: {anchor}")
print(f"Time offset applied to PsychoPy (to NIRS timeline): {offset:.6f} s")
print("\n=== Marker counts (SNIRF vs PsychoPy) ===")
print(counts)

print("\n=== Timing deltas summary ===")
print(quick_stats(merged))

# Optional: save a joined table for manual inspection
out_dir = os.path.dirname(PSY_PATH)
out_csv = os.path.join(out_dir, "events_alignment.csv")
merged.to_csv(out_csv, index=False, encoding="utf-8")
print(f"\nSaved joined events table → {out_csv}")

# Optional: simple sanity prints of first few rows
print("\nFirst 10 SNIRF annotations:")
print(nirs_events.head(10))
print("\nFirst 10 PsychoPy markers (shifted to NIRS time):")
print(psy_events.assign(psy_time_in_nirs_s=psy_events['t_abs'] + offset).head(10))

RuntimeError: No common labels between SNIRF annotations and PsychoPy log.

In [3]:
print("Unique SNIRF labels (raw):")
print(sorted(nirs_events["label"].astype(str).unique())[:60])

print("\nUnique PsychoPy labels (raw):")
print(sorted(pd.read_csv(PSY_PATH)["marker_name"].astype(str).unique()))


Unique SNIRF labels (raw):
['12']

Unique PsychoPy labels (raw):
['ANS_A', 'ANS_B', 'ANS_C', 'ANS_D', 'ANS_E', 'BLK_OFF', 'BLK_ON', 'BLOCK_REST', 'BUTTON_CLICK', 'EXP_END', 'EXP_START', 'ITI', 'PING', 'QNR_ITEM', 'QUESTIONNAIRE_OFF', 'QUESTIONNAIRE_ON', 'Q_OPTIONS_ON', 'Q_STEM_ON', 'Q_TEXT_ON']
